In [1]:

from feupy.utils.config import (
    ObservationConfig, DatasetsConfig, OnOffConfig, SensitivityConfig, StatisticsConfig
)

ImportError: cannot import name 'ObservationConfig' from partially initialized module 'feupy.utils.config' (most likely due to a circular import) (/home/phoenix/Coding/GitHub/feupy/feupy/utils/config.py)

In [1]:
# Licensed under a 3-clause BSD style license - see LICENSE.rst
""" CTAO IRFs class."""

from __future__ import annotations

import os
import logging
from pathlib import Path
from functools import lru_cache
from datetime import datetime, timedelta
from typing import Dict, Tuple, Optional, Iterable

import numpy as np
import pandas as pd
import astropy.units as u
from astropy.coordinates import SkyCoord, AltAz
from astropy.time import Time

from gammapy.irf import load_irf_dict_from_file
from gammapy.data import observatory_locations

try:
    from tqdm import tqdm
except Exception:
    def tqdm(x, **_):
        return x


__all__ = [
    "Irfs",
    "VisibilityCalculator",
    "get_visibility_table_from_position",
]

log = logging.getLogger(__name__)

# -----------------------------------------------------------------------------
# IRF utilities
# -----------------------------------------------------------------------------

IrfsOption = Tuple[str, str, str, str]


def generate_irfs_options() -> Tuple[IrfsOption, ...]:
    arrays = (
        "South",
        "South-SSTSubArray",
        "South-MSTSubArray",
        "North",
        "North-MSTSubArray",
        "North-LSTSubArray",
    )
    azimuths = ("AverageAz", "NorthAz", "SouthAz")
    zeniths = ("20deg", "40deg", "60deg")
    livetimes = ("0.5h", "5h", "50h")

    return tuple(
        (a, az, z, lt)
        for a in arrays
        for az in azimuths
        for z in zeniths
        for lt in livetimes
    )


class Irfs:
    """Handler for CTAO Instrument Response Functions (IRFs)."""

    IRF_VERSION = "prod5 v0.1"
    IRFS_OPTIONS = generate_irfs_options()

    _SITE_ARRAY = {
        "South": "14MSTs37SSTs",
        "South-SSTSubArray": "37SSTs",
        "South-MSTSubArray": "14MSTs",
        "North": "4LSTs09MSTs",
        "North-MSTSubArray": "09MSTs",
        "North-LSTSubArray": "4LSTs",
    }

    _OBS_TIME = {
        "0.5h": "1800s",
        "5h": "18000s",
        "50h": "180000s",
    }

    _DIR_FITS = (
        Path(os.getenv("PYTHONPATH", ""))
        / "data/irfs/cta-prod5-zenodo-v0.1/fits"
    )

    def __init__(self) -> None:
        self._irf_meta_cache: Dict[IrfsOption, Dict] = {}

    # ------------------------------------------------------------------
    # Internal helpers
    # ------------------------------------------------------------------

    @staticmethod
    def _array_label(array_name: str) -> str:
        return array_name.replace("SubArray", "s")

    @staticmethod
    def _observatory_for_option(irfs_opt: IrfsOption):
        return (
            observatory_locations["cta_south"]
            if "South" in irfs_opt[0]
            else observatory_locations["cta_north"]
        )

    @staticmethod
    def _make_file_path(irfs_opt: IrfsOption) -> Path:
        array, az, zen, lt = irfs_opt
        site = array.split("-")[0]

        subdir = f"CTA-Performance-prod5-v0.1-{array}-{zen}.FITS"
        filename = (
            f"Prod5-{site}-{zen}-{az}-"
            f"{Irfs._SITE_ARRAY[array]}.{Irfs._OBS_TIME[lt]}-v0.1.fits.gz"
        )

        return Irfs._DIR_FITS / subdir / filename

    # ------------------------------------------------------------------
    # Labels and names
    # ------------------------------------------------------------------

    @staticmethod
    def get_label(irfs_opt: IrfsOption, which: str = "both") -> str:
        array, az, zen, lt = irfs_opt
        base = f"CTAO {Irfs._array_label(array)}{az.replace('AverageAz', '')}"

        if which == "zenith":
            return f"{base} ({zen})"
        if which == "livetime":
            return f"{base} ({lt})"
        if which == "both":
            return f"{base} ({zen}-{lt})"

        return base

    @staticmethod
    def get_name(irfs_opt: IrfsOption, which: str = "both") -> str:
        array, az, zen, lt = irfs_opt
        base = f"CTAO-{Irfs._array_label(array)}{az.replace('AverageAz', '')}"

        if which == "zenith":
            return f"{base}_{zen}"
        if which == "livetime":
            return f"{base}_{lt}"
        if which == "both":
            return f"{base}_{zen}_{lt}"

        return base

    # ------------------------------------------------------------------
    # Loading
    # ------------------------------------------------------------------

    @lru_cache(maxsize=128)
    def _load_irf_file(self, path: Path) -> dict:
        log.debug("Loading IRF file: %s", path)
        return load_irf_dict_from_file(path)

    def load(self, irfs_opt: IrfsOption) -> Dict:
        if irfs_opt in self._irf_meta_cache:
            return self._irf_meta_cache[irfs_opt]

        path = self._make_file_path(irfs_opt)
        irf = self._load_irf_file(path)

        meta = {
            "irf": irf,
            "label": self.get_label(irfs_opt),
            "name": self.get_name(irfs_opt),
            "obs_location": self._observatory_for_option(irfs_opt),
            "option": irfs_opt,
            "file_path": path,
        }

        self._irf_meta_cache[irfs_opt] = meta
        return meta

    def iter_irfs(
        self,
        arrays: Optional[Iterable[str]] = None,
        azimuths: Optional[Iterable[str]] = None,
        zeniths: Optional[Iterable[str]] = None,
        livetimes: Optional[Iterable[str]] = None,
    ):
        arrays = arrays or {o[0] for o in self.IRFS_OPTIONS}
        azimuths = azimuths or {o[1] for o in self.IRFS_OPTIONS}
        zeniths = zeniths or {o[2] for o in self.IRFS_OPTIONS}
        livetimes = livetimes or {o[3] for o in self.IRFS_OPTIONS}

        for opt in self.IRFS_OPTIONS:
            if (
                opt[0] in arrays
                and opt[1] in azimuths
                and opt[2] in zeniths
                and opt[3] in livetimes
            ):
                try:
                    yield opt, self.load(opt)
                except Exception as e:
                    log.warning("Failed to load IRF %s: %s", opt, e)
                    yield opt, None


# -----------------------------------------------------------------------------
# Visibility utilities
# -----------------------------------------------------------------------------

class VisibilityCalculator:
    """Annual visibility calculator for CTAO sites."""

    ZENITH_RANGES = {
        "20": (10, 30),
        "40": (30, 50),
        "60": (50, 70),
    }

    def __init__(
        self,
        target_position: SkyCoord,
        year: int = 2025,
        time_step_min: int = 30,
        night_start: int = 18,
        night_end: int = 6,
    ):
        self.target_position = target_position
        self.year = year
        self.time_step_min = time_step_min

        hours = list(range(night_start, 24)) + list(range(0, night_end))
        self._time_grid = [
            f"{h:02d}:{m:02d}:00"
            for h in hours
            for m in range(0, 60, time_step_min)
        ]

    def _night_times(self, date: datetime) -> Time:
        date_str = date.strftime("%Y-%m-%d")
        return Time([f"{date_str} {t}" for t in self._time_grid])

    def compute_annual_visibility(
        self,
        observatory: str,
        show_progress: bool = True,
    ) -> Dict[str, float]:
        """
        Compute annual visibility (hours) per zenith-angle bin for one CTAO site.

        Parameters
        ----------
        observatory : {"cta_south", "cta_north"}
            CTAO observatory site.
        show_progress : bool
            Show progress bar (tqdm).

        Returns
        -------
        dict
            Dictionary with zenith-bin labels as keys and visibility in hours as values.
        """
        location = observatory_locations[observatory]
        visibility = {k: 0.0 for k in self.ZENITH_RANGES}
        step_hours = self.time_step_min / 60.0

        start = datetime(self.year, 1, 1)
        end = datetime(self.year + 1, 1, 1)
        days = (end - start).days

        iterator = (
            tqdm(range(days), desc=f"Visibility {observatory}")
            if show_progress
            else range(days)
        )

        for d in iterator:
            times = self._night_times(start + timedelta(days=d))
            altaz = AltAz(obstime=times, location=location)
            target_altaz = self.target_position.transform_to(altaz)
            zenith = 90 * u.deg - target_altaz.alt

            for label, (zmin, zmax) in self.ZENITH_RANGES.items():
                mask = (zenith >= zmin * u.deg) & (zenith < zmax * u.deg)
                visibility[label] += np.count_nonzero(mask) * step_hours

        return visibility

    def to_dataframe(
        self,
        save_path: Optional[str] = None,
        show_progress: bool = True,
    ) -> pd.DataFrame:
        """
        Compute visibility for both CTAO sites and return as a DataFrame.

        Parameters
        ----------
        save_path : str, optional
            Path to save the table (.csv or .tex).
        show_progress : bool
            Show progress bar.

        Returns
        -------
        pandas.DataFrame
        """
        rows = []

        for obs in ("cta_south", "cta_north"):
            vis = self.compute_annual_visibility(obs, show_progress)
            for zbin, hours in vis.items():
                rows.append(
                    {
                        "Observatory": "CTAO South"
                        if obs == "cta_south"
                        else "CTAO North",
                        "Zenith Range (deg)": zbin,
                        "Visibility (hours)": hours,
                    }
                )

        df = pd.DataFrame(rows)

        if save_path:
            path = Path(save_path)
            if path.suffix == ".csv":
                df.to_csv(path, index=False)
            elif path.suffix == ".tex":
                path.write_text(
                    df.to_latex(
                        index=False,
                        float_format=lambda x: f"{x:.2f}",
                        caption="Annual CTAO visibility per zenith range.",
                        label="tab:ctao_visibility",
                    )
                )

        return df


def get_visibility_table_from_position(
    target_position: SkyCoord,
    year: int = 2025,
    time_step_min: int = 30,
    save_path: Optional[str] = None,
) -> pd.DataFrame:
    """
    Convenience function to compute CTAO annual visibility from a SkyCoord position.
    """
    calc = VisibilityCalculator(
        target_position=target_position,
        year=year,
        time_step_min=time_step_min,
    )
    return calc.to_dataframe(save_path=save_path)






